# 03 - Submission

Tujuan: membuat file submission dari model akhir (HistGradientBoosting + One-vs-One, parameter hasil tuning), memvalidasi formatnya, dan memeriksa apakah distribusi prediksi masuk akal.

Prasyarat: `data/train.csv`, `data/test.csv`, dan `models/model.joblib` (hasil `python -m scripts.train --model hgb --strategy ovo --params models/best_params.json --skip-cv`).

In [ ]:
import os, sys
from pathlib import Path

# Notebook ada di notebooks/, sedangkan scripts/ dan data/ ada di root repo
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from scripts.predict import make_submission
from scripts.preprocess import ID_COL, TARGET, decode_target, load_data, split_features_target

MODEL_PATH = "models/model.joblib"
assert Path(MODEL_PATH).exists(), "Model belum ada. Jalankan scripts.train dulu (lihat README)."
train, test = load_data("data")

## 1. Buat submission

In [ ]:
sub = make_submission(MODEL_PATH, "data/test.csv", "submission.csv")
sub.head()

## 2. Validasi format
Kaggle meminta kolom `building_id` dan `damage_grade` (angka 1-5) dengan urutan baris sama seperti `test.csv`.

In [ ]:
assert list(sub.columns) == [ID_COL, TARGET]
assert len(sub) == len(test)
assert sub[ID_COL].equals(test[ID_COL])
assert set(sub[TARGET].unique()) <= {1, 2, 3, 4, 5}
print("Format submission valid:", sub.shape)

## 3. Distribusi prediksi vs distribusi train

In [ ]:
_, y_train = split_features_target(train)
dist = pd.concat({
    "train (asli)": pd.Series(decode_target(y_train.values)).value_counts(normalize=True),
    "test (prediksi)": sub[TARGET].value_counts(normalize=True),
}, axis=1).sort_index().round(4)
display(dist)

ax = dist.plot.bar(figsize=(7, 4), rot=0, title="Proporsi kelas: train vs prediksi test")
ax.set_xlabel("damage_grade")
plt.tight_layout()
plt.savefig("reports/figures/pred_vs_train_distribution.png", dpi=150)

**Interpretasi distribusi prediksi:** Pada validasi hold-out, rasio recall/presisi tiap kelas menunjukkan model cenderung menebak Grade 1 lebih sering dari proporsi aslinya (0,764/0,552, sekitar 1,4 kali) dan Grade 5 lebih jarang (0,656/0,761, sekitar 0,86 kali). Maka pada data test diharapkan proporsi prediksi Grade 1 sekitar 14% (asli 10%) dan Grade 5 sekitar 31% (asli 36%), sedangkan Grade 2 sampai 4 hampir sama dengan proporsi aslinya. Pergeseran ini disengaja: `class_weight="balanced"` membuat model lebih berani memprediksi kelas minoritas agar Macro F1 naik, dengan konsekuensi presisi kelas tersebut turun. Bandingkan dengan tabel di atas; jika pergeserannya jauh berbeda dari perkiraan ini, periksa kembali apakah model yang dipakai sudah benar.

## 4. Hasil leaderboard

| | Macro F1 |
|---|---|
| CV tuning (subsampel 100 ribu) | 0,4924 |
| Hold-out (20% data train penuh) | 0,5134 |
| Public leaderboard Kaggle | 0,51329 (peringkat 21 saat disubmit) |

Skor public hampir sama dengan hold-out, artinya validasi lokal dapat dipercaya dan model tidak overfit ke data latih. Tidak ada tanda kebocoran data: seluruh preprocessing berada di dalam `Pipeline`, sehingga statistik hanya dipelajari dari data latih tiap fold.